# Mid-Semester Report: TST-Enhanced TransApp for Energy Time Series Classification

**Author**: [Your Name]  
**Date**: December 2024  
**Course**: [Course Name]  

## Executive Summary

This report presents the implementation and evaluation of Time Series Transformer (TST) enhancements to the TransApp architecture for energy time series classification and appliance detection. We conducted comprehensive experiments comparing standard TransApp with TST-enhanced versions across multiple datasets and configurations.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import glob
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Setup paths and imports
root = Path(os.getcwd()).resolve().parents[0]
sys.path.append(str(root))

# Set up plotting style
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 10
plt.rcParams['ytick.labelsize'] = 10
plt.rcParams['legend.fontsize'] = 11
plt.rcParams['figure.titlesize'] = 16

# Color palette for consistency
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#592E83']
sns.set_palette(colors)

print("📊 Mid-Semester Report Analysis Setup Complete")
print(f"Working directory: {root}")

## 1. Research Objectives and Methodology

### 1.1 Objectives
- **Primary**: Enhance TransApp architecture with Time Series Transformer (TST) components
- **Secondary**: Evaluate performance improvements on energy appliance detection tasks
- **Tertiary**: Compare different normalization strategies (BatchNorm vs LayerNorm)

### 1.2 Methodology
- **Baseline**: Standard TransApp architecture
- **Enhancement**: TST-enhanced TransApp with improved attention mechanisms
- **Evaluation**: Multiple appliance detection cases on CER and COMSTOCK datasets
- **Metrics**: Accuracy, F1-Score, ROC-AUC for both subsequence and full time series evaluation

In [ ]:
def load_experiment_results():
    """Load all experiment results from various directories"""
    
    results_data = {
        'pretraining': [],
        'classification': [],
        'comparison': []
    }
    
    # Load TST classification results - focus on actual results
    classif_dir = root / 'results' / 'TransAppResults_TST'
    print(f"🔍 Looking for results in: {classif_dir}")
    
    if classif_dir.exists():
        # Load from both None and Embed directories
        for embed_type in ['None', 'Embed']:
            embed_dir = classif_dir / embed_type
            if embed_dir.exists():
                print(f"📁 Searching in {embed_type} directory...")
                for json_file in embed_dir.rglob('*_results.json'):
                    try:
                        with open(json_file, 'r') as f:
                            data = json.load(f)
                            data['source'] = 'classification'
                            data['embed_type_dir'] = embed_type
                            data['file_path'] = str(json_file)
                            results_data['classification'].append(data)
                            print(f"✅ Loaded: {json_file.name}")
                    except Exception as e:
                        print(f"⚠️ Could not load {json_file}: {e}")
                
                # Also load comprehensive results
                for json_file in embed_dir.rglob('comprehensive_*.json'):
                    try:
                        with open(json_file, 'r') as f:
                            data = json.load(f)
                            data['source'] = 'comprehensive'
                            data['embed_type_dir'] = embed_type
                            data['file_path'] = str(json_file)
                            results_data['classification'].append(data)
                            print(f"✅ Loaded comprehensive: {json_file.name}")
                    except Exception as e:
                        print(f"⚠️ Could not load {json_file}: {e}")
    
    # Load TST pretraining results
    pretrain_dir = root / 'results' / 'TransAppPretrained_TST'
    if pretrain_dir.exists():
        for json_file in pretrain_dir.rglob('*.json'):
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    data['source'] = 'pretraining'
                    data['file_path'] = str(json_file)
                    results_data['pretraining'].append(data)
                    print(f"✅ Loaded pretraining: {json_file.name}")
            except Exception as e:
                print(f"⚠️ Could not load {json_file}: {e}")
    
    # Load architecture comparison results
    comparison_dir = root / 'results' / 'Architecture_Comparison'
    if comparison_dir.exists():
        for json_file in comparison_dir.rglob('*.json'):
            try:
                with open(json_file, 'r') as f:
                    data = json.load(f)
                    data['source'] = 'comparison'
                    data['file_path'] = str(json_file)
                    results_data['comparison'].append(data)
                    print(f"✅ Loaded comparison: {json_file.name}")
            except Exception as e:
                print(f"⚠️ Could not load {json_file}: {e}")
    
    return results_data

# Load all available results
results_data = load_experiment_results()

print(f"\n📁 Loaded Results Summary:")
print(f"   Pretraining experiments: {len(results_data['pretraining'])}")
print(f"   Classification experiments: {len(results_data['classification'])}")
print(f"   Architecture comparisons: {len(results_data['comparison'])}")

# Display detailed breakdown of classification results
classification_by_type = {}
for result in results_data['classification']:
    embed_type = result.get('embed_type_dir', 'unknown')
    if embed_type not in classification_by_type:
        classification_by_type[embed_type] = 0
    classification_by_type[embed_type] += 1

print(f"\n📊 Classification Results Breakdown:")
for embed_type, count in classification_by_type.items():
    print(f"   {embed_type}: {count} files")

## 2. Load and Process Actual TST Results

Extract performance metrics from your actual experimental results for both None and Embed encoding cases.

In [ ]:
def extract_classification_metrics_from_tst_results(results_data):
    """Extract classification metrics from actual TST results"""
    
    metrics_data = []
    
    for result in results_data['classification']:
        try:
            embed_type_dir = result.get('embed_type_dir', 'unknown')
            
            # Handle comprehensive results format
            if 'all_results' in result:
                experiment_info = result.get('experiment_info', {})
                base_info = {
                    'case_name': experiment_info.get('case_name', 'unknown'),
                    'model_name': experiment_info.get('model_name', 'unknown'),
                    'dim_model': experiment_info.get('dim_model', 0),
                    'norm_type': experiment_info.get('norm_type', 'unknown'),
                    'dataset_type': experiment_info.get('dataset_type', 'unknown'),
                    'epochs': experiment_info.get('epochs', 0),
                    'embed_type_dir': embed_type_dir
                }
                
                for seed_result in result['all_results']:
                    if 'results' in seed_result and 'quantile_metrics' in seed_result['results']:
                        metrics = seed_result['results']['quantile_metrics']
                        row = base_info.copy()
                        row.update({
                            'embed_name': seed_result.get('embed_name', embed_type_dir),
                            'random_seed': seed_result.get('random_seed', 0),
                            'accuracy': metrics.get('accuracy', 0),
                            'f1_score': metrics.get('f1_score', 0),
                            'precision': metrics.get('precision', 0),
                            'recall': metrics.get('recall', 0),
                            'roc_auc': metrics.get('roc_auc', 0)
                        })
                        metrics_data.append(row)
            
            # Handle single result format
            elif 'results' in result and 'quantile_metrics' in result['results']:
                metrics = result['results']['quantile_metrics']
                config = result.get('configuration', {})
                
                row = {
                    'case_name': result.get('case_name', 'unknown'),
                    'model_name': result.get('model_name', 'unknown'),
                    'dim_model': result.get('dim_model', 0),
                    'norm_type': config.get('norm_type', 'unknown'),
                    'dataset_type': config.get('dataset_type', 'unknown'),
                    'embed_type_dir': embed_type_dir,
                    'embed_name': embed_type_dir,
                    'random_seed': config.get('random_seed', 0),
                    'epochs': 0,
                    'accuracy': metrics.get('accuracy', 0),
                    'f1_score': metrics.get('f1_score', 0),
                    'precision': metrics.get('precision', 0),
                    'recall': metrics.get('recall', 0),
                    'roc_auc': metrics.get('roc_auc', 0)
                }
                metrics_data.append(row)
                
        except Exception as e:
            print(f"⚠️ Error processing result: {e}")
            continue
    
    return pd.DataFrame(metrics_data)

# Extract actual metrics from your TST results
df_metrics = extract_classification_metrics_from_tst_results(results_data)
print(f"📊 Extracted {len(df_metrics)} actual TST classification results")

if not df_metrics.empty:
    print(f"\n📋 Data Summary:")
    print(f"   Cases: {sorted(df_metrics['case_name'].unique())}")
    print(f"   Models: {sorted(df_metrics['model_name'].unique())}")
    print(f"   Dimensions: {sorted(df_metrics['dim_model'].unique())}")
    print(f"   Embedding types: {sorted(df_metrics['embed_name'].unique())}")
    print(f"   Normalization types: {sorted(df_metrics['norm_type'].unique())}")
    
    # Show sample of actual data
    print(f"\n📈 Sample of actual data:")
    display_cols = ['case_name', 'model_name', 'embed_name', 'f1_score', 'accuracy', 'roc_auc']
    available_cols = [col for col in display_cols if col in df_metrics.columns]
    print(df_metrics[available_cols].head(10))
    
    # Statistics by embedding type
    print(f"\n📊 Performance by Embedding Type:")
    if 'embed_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        embed_stats = df_metrics.groupby('embed_name')['f1_score'].agg(['count', 'mean', 'std']).round(4)
        print(embed_stats)
        
else:
    print("⚠️ No classification metrics found in actual results.")

## 3. Performance Analysis Using Actual TST Results

Create comprehensive performance analysis based on your actual experimental results from both None and Embed encoding cases.

In [ ]:
def create_actual_performance_plots(df_metrics):
    """Create performance plots using actual TST experimental results"""
    
    if df_metrics.empty:
        print("⚠️ No actual data available for plotting")
        return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Performance by Case and Model Type
    if 'case_name' in df_metrics.columns and 'model_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        # Calculate mean F1-score by case and model
        perf_by_case_model = df_metrics.groupby(['case_name', 'model_name'])['f1_score'].mean().reset_index()
        
        if not perf_by_case_model.empty:
            pivot_data = perf_by_case_model.pivot(index='case_name', columns='model_name', values='f1_score')
            
            if not pivot_data.empty:
                pivot_data.plot(kind='bar', ax=ax1, color=colors[:len(pivot_data.columns)], width=0.8)
                ax1.set_title('F1-Score by Case and Model Type (Actual Results)')
                ax1.set_ylabel('F1-Score')
                ax1.set_xlabel('Case')
                ax1.legend(title='Model Type', bbox_to_anchor=(1.05, 1), loc='upper left')
                ax1.tick_params(axis='x', rotation=45)
                ax1.grid(axis='y', alpha=0.3)
                
                # Add value labels on bars
                for container in ax1.containers:
                    ax1.bar_label(container, fmt='%.3f', rotation=90, fontsize=8)
    
    # 2. Performance by Embedding Type (None vs Embed)
    if 'embed_name' in df_metrics.columns:
        embed_performance = df_metrics.groupby('embed_name')[['accuracy', 'f1_score', 'roc_auc']].mean()
        
        if not embed_performance.empty:
            embed_performance.plot(kind='bar', ax=ax2, color=colors[:3], width=0.7)
            ax2.set_title('Performance by Embedding Type (Actual Results)')
            ax2.set_ylabel('Score')
            ax2.set_xlabel('Embedding Type')
            ax2.legend(title='Metric')
            ax2.tick_params(axis='x', rotation=0)
            ax2.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for container in ax2.containers:
                ax2.bar_label(container, fmt='%.3f', fontsize=9)
    
    # 3. Model Performance Distribution
    if 'f1_score' in df_metrics.columns and 'model_name' in df_metrics.columns:
        model_names = df_metrics['model_name'].unique()
        
        # Create box plot
        f1_data_by_model = [df_metrics[df_metrics['model_name'] == model]['f1_score'].values 
                           for model in model_names]
        
        box_plot = ax3.boxplot(f1_data_by_model, labels=model_names, patch_artist=True)
        
        # Color the boxes
        for patch, color in zip(box_plot['boxes'], colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        
        ax3.set_title('F1-Score Distribution by Model (Actual Results)')
        ax3.set_ylabel('F1-Score')
        ax3.tick_params(axis='x', rotation=45)
        ax3.grid(axis='y', alpha=0.3)
    
    # 4. Embedding Type Comparison by Case
    if 'case_name' in df_metrics.columns and 'embed_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        embed_case_perf = df_metrics.groupby(['case_name', 'embed_name'])['f1_score'].mean().reset_index()
        
        if not embed_case_perf.empty:
            pivot_embed = embed_case_perf.pivot(index='case_name', columns='embed_name', values='f1_score')
            
            if not pivot_embed.empty and len(pivot_embed.columns) > 1:
                # Calculate improvement
                if 'None' in pivot_embed.columns and 'Embed' in pivot_embed.columns:
                    improvement = ((pivot_embed['Embed'] - pivot_embed['None']) / pivot_embed['None'] * 100).fillna(0)
                    
                    bars = ax4.bar(range(len(improvement)), improvement.values, 
                                  color=[colors[1] if x > 0 else colors[3] for x in improvement.values],
                                  alpha=0.8)
                    
                    ax4.set_title('Embedding Improvement by Case (% F1-Score)')
                    ax4.set_ylabel('Improvement (%)')
                    ax4.set_xlabel('Case')
                    ax4.set_xticks(range(len(improvement)))
                    ax4.set_xticklabels(improvement.index, rotation=45)
                    ax4.grid(axis='y', alpha=0.3)
                    ax4.axhline(y=0, color='black', linestyle='-', alpha=0.5)
                    
                    # Add value labels
                    for i, (bar, val) in enumerate(zip(bars, improvement.values)):
                        height = bar.get_height()
                        ax4.text(bar.get_x() + bar.get_width()/2., 
                                height + (0.5 if height >= 0 else -0.5),
                                f'{val:.1f}%', ha='center', 
                                va='bottom' if height >= 0 else 'top', fontweight='bold')
                else:
                    # Just plot the embedding comparison
                    pivot_embed.plot(kind='bar', ax=ax4, color=colors[:len(pivot_embed.columns)])
                    ax4.set_title('F1-Score by Case and Embedding Type')
                    ax4.set_ylabel('F1-Score')
                    ax4.tick_params(axis='x', rotation=45)
                    ax4.legend(title='Embedding Type')
    
    plt.tight_layout()
    plt.savefig(root / 'results' / 'actual_tst_performance_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create performance plots with actual data
create_actual_performance_plots(df_metrics)

## 4. Statistical Analysis of Actual Results

Analyze the statistical significance and trends in your actual TST experimental data.

In [ ]:
def create_statistical_analysis_actual_data(df_metrics):
    """Create statistical analysis using actual TST results"""
    
    if df_metrics.empty:
        print("⚠️ No data available for statistical analysis")
        return
    
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(16, 12))
    
    # 1. Performance Statistics by Model
    if 'model_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        model_stats = df_metrics.groupby('model_name')['f1_score'].agg(['mean', 'std', 'count']).reset_index()
        
        bars = ax1.bar(range(len(model_stats)), model_stats['mean'], 
                      yerr=model_stats['std'], capsize=5, color=colors[:len(model_stats)], alpha=0.8)
        
        ax1.set_title('F1-Score by Model Type (Mean ± Std)')
        ax1.set_ylabel('F1-Score')
        ax1.set_xlabel('Model Type')
        ax1.set_xticks(range(len(model_stats)))
        ax1.set_xticklabels(model_stats['model_name'], rotation=45)
        ax1.grid(axis='y', alpha=0.3)
        
        # Add value labels
        for i, (bar, mean_val, std_val) in enumerate(zip(bars, model_stats['mean'], model_stats['std'])):
            height = bar.get_height()
            ax1.text(bar.get_x() + bar.get_width()/2., height + std_val + 0.01,
                    f'{mean_val:.3f}±{std_val:.3f}', ha='center', va='bottom', fontsize=9)
    
    # 2. Normalization Strategy Comparison
    if 'norm_type' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        norm_comparison = df_metrics.groupby('norm_type')['f1_score'].agg(['mean', 'std', 'count']).reset_index()
        
        if len(norm_comparison) > 1:
            bars = ax2.bar(range(len(norm_comparison)), norm_comparison['mean'],
                          yerr=norm_comparison['std'], capsize=5, 
                          color=colors[:len(norm_comparison)], alpha=0.8)
            
            ax2.set_title('Performance by Normalization Type')
            ax2.set_ylabel('F1-Score')
            ax2.set_xlabel('Normalization Type')
            ax2.set_xticks(range(len(norm_comparison)))
            ax2.set_xticklabels(norm_comparison['norm_type'])
            ax2.grid(axis='y', alpha=0.3)
            
            # Add value labels
            for bar, mean_val in zip(bars, norm_comparison['mean']):
                height = bar.get_height()
                ax2.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{mean_val:.3f}', ha='center', va='bottom', fontweight='bold')
    
    # 3. Model Dimension Analysis
    if 'dim_model' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        dim_performance = df_metrics.groupby('dim_model')['f1_score'].agg(['mean', 'std']).reset_index()
        
        if not dim_performance.empty:
            ax3.errorbar(dim_performance['dim_model'], dim_performance['mean'], 
                        yerr=dim_performance['std'], marker='o', capsize=5, 
                        color=colors[0], linewidth=2, markersize=8)
            
            ax3.set_title('F1-Score vs Model Dimension')
            ax3.set_xlabel('Model Dimension')
            ax3.set_ylabel('F1-Score (mean ± std)')
            ax3.grid(True, alpha=0.3)
            
            # Add value labels
            for dim, mean_val, std_val in zip(dim_performance['dim_model'], 
                                            dim_performance['mean'], 
                                            dim_performance['std']):
                ax3.annotate(f'{mean_val:.3f}±{std_val:.3f}', 
                           (dim, mean_val), textcoords="offset points", 
                           xytext=(0,10), ha='center', fontsize=9)
    
    # 4. Embedding Type Statistical Comparison
    if 'embed_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
        embed_stats = df_metrics.groupby('embed_name')['f1_score'].agg(['mean', 'std', 'count']).reset_index()
        
        bars = ax4.bar(range(len(embed_stats)), embed_stats['mean'],
                      yerr=embed_stats['std'], capsize=5, 
                      color=colors[:len(embed_stats)], alpha=0.8)
        
        ax4.set_title('Performance by Embedding Type')
        ax4.set_ylabel('F1-Score')
        ax4.set_xlabel('Embedding Type')
        ax4.set_xticks(range(len(embed_stats)))
        ax4.set_xticklabels(embed_stats['embed_name'])
        ax4.grid(axis='y', alpha=0.3)
        
        # Add statistical significance test if we have both None and Embed
        if len(embed_stats) == 2:
            from scipy import stats
            
            none_data = df_metrics[df_metrics['embed_name'] == 'None']['f1_score'].dropna()
            embed_data = df_metrics[df_metrics['embed_name'] == 'Embed']['f1_score'].dropna()
            
            if len(none_data) > 1 and len(embed_data) > 1:
                t_stat, p_value = stats.ttest_ind(none_data, embed_data)
                
                significance = ""
                if p_value < 0.001:
                    significance = "***"
                elif p_value < 0.01:
                    significance = "**"
                elif p_value < 0.05:
                    significance = "*"
                else:
                    significance = "ns"
                
                # Add significance annotation
                max_height = max(embed_stats['mean'] + embed_stats['std'])
                ax4.text(0.5, max_height + 0.02, 
                        f't-test: p={p_value:.4f} {significance}', 
                        ha='center', va='bottom', fontweight='bold',
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
        
        # Add value labels
        for bar, mean_val, std_val in zip(bars, embed_stats['mean'], embed_stats['std']):
            height = bar.get_height()
            ax4.text(bar.get_x() + bar.get_width()/2., height + std_val + 0.01,
                    f'{mean_val:.3f}±{std_val:.3f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.savefig(root / 'results' / 'actual_tst_statistical_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create statistical analysis with actual data
create_statistical_analysis_actual_data(df_metrics)

## 5. Comprehensive Results Summary Table

Generate a comprehensive summary table from your actual TST experimental results.

In [ ]:
def create_actual_results_summary_table(df_metrics):
    """Create comprehensive results summary table from actual data"""
    
    if df_metrics.empty:
        print("⚠️ No data available for summary table")
        return
    
    print("📊 COMPREHENSIVE SUMMARY OF ACTUAL TST RESULTS")
    print("="*60)
    
    # Overall statistics
    if 'f1_score' in df_metrics.columns:
        print(f"\n📈 Overall Performance Statistics:")
        overall_stats = df_metrics[['accuracy', 'f1_score', 'precision', 'recall', 'roc_auc']].describe()
        print(overall_stats.round(4))
    
    # Performance by embedding type
    if 'embed_name' in df_metrics.columns:
        print(f"\n🎯 Performance by Embedding Type:")
        embed_perf = df_metrics.groupby('embed_name')[['accuracy', 'f1_score', 'roc_auc']].agg(['mean', 'std', 'count']).round(4)
        print(embed_perf)
    
    # Performance by model type
    if 'model_name' in df_metrics.columns:
        print(f"\n🏗️ Performance by Model Type:")
        model_perf = df_metrics.groupby('model_name')[['accuracy', 'f1_score', 'roc_auc']].agg(['mean', 'std', 'count']).round(4)
        print(model_perf)
    
    # Create visual summary table
    if 'case_name' in df_metrics.columns and 'embed_name' in df_metrics.columns:
        # Create summary for visualization
        summary_data = []
        
        for case in df_metrics['case_name'].unique():
            case_data = df_metrics[df_metrics['case_name'] == case]
            
            for embed_type in case_data['embed_name'].unique():
                embed_case_data = case_data[case_data['embed_name'] == embed_type]
                
                if not embed_case_data.empty and 'f1_score' in embed_case_data.columns:
                    row = {
                        'Case': case,
                        'Embedding': embed_type,
                        'Count': len(embed_case_data),
                        'F1_Mean': embed_case_data['f1_score'].mean(),
                        'F1_Std': embed_case_data['f1_score'].std(),
                        'Accuracy_Mean': embed_case_data['accuracy'].mean() if 'accuracy' in embed_case_data.columns else np.nan,
                        'ROC_AUC_Mean': embed_case_data['roc_auc'].mean() if 'roc_auc' in embed_case_data.columns else np.nan
                    }
                    summary_data.append(row)
        
        if summary_data:
            summary_df = pd.DataFrame(summary_data)
            
            # Create styled table plot
            fig, ax = plt.subplots(figsize=(16, 10))
            ax.axis('tight')
            ax.axis('off')
            
            # Prepare table data
            table_data = []
            headers = ['Case', 'Embedding', 'Runs', 'F1-Score', 'Accuracy', 'ROC-AUC']
            
            for _, row in summary_df.iterrows():
                table_row = [
                    row['Case'],
                    row['Embedding'],
                    f"{int(row['Count'])}",
                    f"{row['F1_Mean']:.3f} ± {row['F1_Std']:.3f}" if not pd.isna(row['F1_Std']) else f"{row['F1_Mean']:.3f}",
                    f"{row['Accuracy_Mean']:.3f}" if not pd.isna(row['Accuracy_Mean']) else "N/A",
                    f"{row['ROC_AUC_Mean']:.3f}" if not pd.isna(row['ROC_AUC_Mean']) else "N/A"
                ]
                table_data.append(table_row)
            
            # Create table
            table = ax.table(cellText=table_data, colLabels=headers,
                           cellLoc='center', loc='center',
                           colWidths=[0.15, 0.12, 0.08, 0.2, 0.15, 0.15])
            
            # Style the table
            table.auto_set_font_size(False)
            table.set_fontsize(10)
            table.scale(1.2, 2.5)
            
            # Color code headers
            for i in range(len(headers)):
                table[(0, i)].set_facecolor('#2E86AB')
                table[(0, i)].set_text_props(weight='bold', color='white')
            
            # Color code by embedding type
            for i in range(1, len(table_data) + 1):
                embedding_type = table_data[i-1][1]  # Embedding column
                if embedding_type == 'None':
                    color = '#F0F8FF'  # Light blue
                elif embedding_type == 'Embed':
                    color = '#F0FFF0'  # Light green
                else:
                    color = '#FFFFFF'  # White
                
                for j in range(len(headers)):
                    table[(i, j)].set_facecolor(color)
            
            plt.title('Actual TST Experimental Results Summary', 
                     fontsize=16, fontweight='bold', pad=20)
            plt.savefig(root / 'results' / 'actual_tst_results_summary_table.png', dpi=300, bbox_inches='tight')
            plt.show()
            
            return summary_df
    
    return None

# Create results summary table with actual data
summary_table = create_actual_results_summary_table(df_metrics)
if summary_table is not None:
    print("✅ Actual results summary table created successfully")

## 6. Key Findings from Actual TST Experiments

Extract and analyze key findings from your actual experimental results.

In [ ]:
def generate_actual_key_findings(df_metrics):
    """Generate key findings based on actual TST experimental results"""
    
    print("🔍 KEY FINDINGS FROM ACTUAL TST EXPERIMENTS")
    print("="*60)
    
    findings = []
    
    if not df_metrics.empty:
        # 1. Overall performance
        if 'f1_score' in df_metrics.columns:
            avg_f1 = df_metrics['f1_score'].mean()
            std_f1 = df_metrics['f1_score'].std()
            min_f1 = df_metrics['f1_score'].min()
            max_f1 = df_metrics['f1_score'].max()
            
            findings.append(f"📊 Overall F1-Score: {avg_f1:.3f} ± {std_f1:.3f} (range: {min_f1:.3f} to {max_f1:.3f})")
        
        # 2. Embedding type comparison
        if 'embed_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            embed_performance = df_metrics.groupby('embed_name')['f1_score'].agg(['mean', 'std', 'count'])
            
            if 'None' in embed_performance.index and 'Embed' in embed_performance.index:
                none_perf = embed_performance.loc['None', 'mean']
                embed_perf = embed_performance.loc['Embed', 'mean']
                improvement = ((embed_perf - none_perf) / none_perf) * 100
                
                findings.append(f"🎯 Embedding Impact: {embed_perf:.3f} vs {none_perf:.3f} ({improvement:+.1f}% change)")
                
                if improvement > 0:
                    findings.append(f"✅ Temporal embeddings show improvement over no embeddings")
                else:
                    findings.append(f"❌ Temporal embeddings show degradation compared to no embeddings")
        
        # 3. Model comparison
        if 'model_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            model_performance = df_metrics.groupby('model_name')['f1_score'].agg(['mean', 'count'])
            
            if len(model_performance) > 1:
                best_model = model_performance['mean'].idxmax()
                best_score = model_performance['mean'].max()
                worst_model = model_performance['mean'].idxmin()
                worst_score = model_performance['mean'].min()
                
                findings.append(f"🏆 Best model: {best_model} (F1: {best_score:.3f})")
                findings.append(f"📉 Baseline model: {worst_model} (F1: {worst_score:.3f})")
                
                if best_score > worst_score:
                    improvement = ((best_score - worst_score) / worst_score) * 100
                    findings.append(f"📈 Best model improvement: {improvement:.1f}% over baseline")
        
        # 4. Normalization analysis
        if 'norm_type' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            norm_performance = df_metrics.groupby('norm_type')['f1_score'].mean()
            
            if len(norm_performance) > 1:
                best_norm = norm_performance.idxmax()
                findings.append(f"⚙️ Best normalization strategy: {best_norm}")
        
        # 5. Case-specific insights
        if 'case_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            case_performance = df_metrics.groupby('case_name')['f1_score'].mean().sort_values()
            
            if len(case_performance) > 1:
                hardest_case = case_performance.index[0]
                easiest_case = case_performance.index[-1]
                findings.append(f"🎯 Most challenging case: {hardest_case} (F1: {case_performance.iloc[0]:.3f})")
                findings.append(f"✅ Best performing case: {easiest_case} (F1: {case_performance.iloc[-1]:.3f})")
        
        # 6. Model dimension analysis
        if 'dim_model' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            dim_performance = df_metrics.groupby('dim_model')['f1_score'].mean()
            
            if len(dim_performance) > 1:
                best_dim = dim_performance.idxmax()
                best_dim_score = dim_performance.max()
                findings.append(f"🔧 Optimal model dimension: {best_dim} (F1: {best_dim_score:.3f})")
        
        # 7. Statistical significance
        if 'embed_name' in df_metrics.columns and len(df_metrics['embed_name'].unique()) == 2:
            try:
                from scipy import stats
                
                none_data = df_metrics[df_metrics['embed_name'] == 'None']['f1_score'].dropna()
                embed_data = df_metrics[df_metrics['embed_name'] == 'Embed']['f1_score'].dropna()
                
                if len(none_data) > 1 and len(embed_data) > 1:
                    t_stat, p_value = stats.ttest_ind(none_data, embed_data)
                    
                    if p_value < 0.05:
                        findings.append(f"📊 Embedding effect is statistically significant (p={p_value:.4f})")
                    else:
                        findings.append(f"📊 Embedding effect is not statistically significant (p={p_value:.4f})")
            except:
                findings.append(f"📊 Statistical significance test could not be performed")
    
    # Display findings
    for i, finding in enumerate(findings, 1):
        print(f"{i}. {finding}")
    
    if not findings:
        print("⚠️ Insufficient data for generating insights")
    
    # Summary statistics
    print(f"\n📝 EXPERIMENT SUMMARY")
    print(f"Total experiments analyzed: {len(df_metrics)}")
    print(f"Unique cases tested: {df_metrics['case_name'].nunique() if 'case_name' in df_metrics.columns else 0}")
    print(f"Unique models tested: {df_metrics['model_name'].nunique() if 'model_name' in df_metrics.columns else 0}")
    print(f"Embedding types: {list(df_metrics['embed_name'].unique()) if 'embed_name' in df_metrics.columns else 'N/A'}")
    print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    
    return findings

# Generate key findings from actual data
actual_findings = generate_actual_key_findings(df_metrics)

## 7. Save Comprehensive Analysis Report

Save all analysis results and findings from your actual TST experiments to comprehensive report files.

In [ ]:
def save_actual_analysis_report(df_metrics, findings):
    """Save comprehensive analysis report based on actual TST results"""
    
    report_data = {
        'timestamp': datetime.now().isoformat(),
        'summary': {
            'total_experiments': len(df_metrics),
            'unique_cases': df_metrics['case_name'].nunique() if 'case_name' in df_metrics.columns else 0,
            'unique_models': df_metrics['model_name'].nunique() if 'model_name' in df_metrics.columns else 0,
            'embedding_types': list(df_metrics['embed_name'].unique()) if 'embed_name' in df_metrics.columns else [],
            'normalization_types': list(df_metrics['norm_type'].unique()) if 'norm_type' in df_metrics.columns else []
        },
        'key_findings': findings,
        'statistics': {}
    }
    
    if not df_metrics.empty:
        # Add statistical summaries
        metrics_cols = ['accuracy', 'f1_score', 'precision', 'recall', 'roc_auc']
        available_metrics = [col for col in metrics_cols if col in df_metrics.columns]
        
        if available_metrics:
            stats_dict = df_metrics[available_metrics].describe().to_dict()
            report_data['statistics']['performance_metrics'] = stats_dict
        
        # Add performance by embedding type
        if 'embed_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            embed_stats = df_metrics.groupby('embed_name')['f1_score'].agg(['mean', 'std', 'count']).to_dict()
            report_data['statistics']['embedding_performance'] = embed_stats
        
        # Add performance by model type
        if 'model_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            model_stats = df_metrics.groupby('model_name')['f1_score'].agg(['mean', 'std', 'count']).to_dict()
            report_data['statistics']['model_performance'] = model_stats
        
        # Add performance by case
        if 'case_name' in df_metrics.columns and 'f1_score' in df_metrics.columns:
            case_stats = df_metrics.groupby('case_name')['f1_score'].agg(['mean', 'std', 'count']).to_dict()
            report_data['statistics']['case_performance'] = case_stats
    
    # Save JSON report
    report_file = root / 'results' / f'actual_tst_analysis_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
    with open(report_file, 'w') as f:
        json.dump(report_data, f, indent=2, default=str)
    
    print(f"💾 Comprehensive analysis report saved to: {report_file}")
    
    # Create markdown summary
    md_file = root / 'results' / f'actual_tst_analysis_summary_{datetime.now().strftime("%Y%m%d_%H%M%S")}.md'
    with open(md_file, 'w') as f:
        f.write(f"# Actual TST-Enhanced TransApp Analysis Report\n\n")
        f.write(f"**Generated:** {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write(f"## Summary\n")
        f.write(f"- Total experiments: {report_data['summary']['total_experiments']}\n")
        f.write(f"- Unique test cases: {report_data['summary']['unique_cases']}\n")
        f.write(f"- Unique models: {report_data['summary']['unique_models']}\n")
        f.write(f"- Embedding types: {report_data['summary']['embedding_types']}\n")
        f.write(f"- Normalization types: {report_data['summary']['normalization_types']}\n\n")
        
        f.write(f"## Key Findings\n")
        for i, finding in enumerate(findings, 1):
            f.write(f"{i}. {finding}\n")
        
        if not df_metrics.empty and 'f1_score' in df_metrics.columns:
            f.write(f"\n## Performance Summary\n")
            f.write(f"- Overall F1-Score: {df_metrics['f1_score'].mean():.3f} ± {df_metrics['f1_score'].std():.3f}\n")
            f.write(f"- Best F1-Score: {df_metrics['f1_score'].max():.3f}\n")
            f.write(f"- Worst F1-Score: {df_metrics['f1_score'].min():.3f}\n")
        
        f.write(f"\n## Generated Plots\n")
        f.write(f"- actual_tst_performance_analysis.png\n")
        f.write(f"- actual_tst_statistical_analysis.png\n")
        f.write(f"- actual_tst_results_summary_table.png\n")
    
    print(f"📄 Markdown summary saved to: {md_file}")
    
    return report_file, md_file

# Save comprehensive report with actual data
if not df_metrics.empty:
    report_files = save_actual_analysis_report(df_metrics, actual_findings)
    print("\n✅ Analysis of actual TST results complete!")
    print("🎯 All results and plots have been generated based on your actual experimental data.")
else:
    print("\n⚠️ No actual experimental data found.")
    print("Please ensure your TST experiments have been run and results are saved in:")
    print(f"   {root}/results/TransAppResults_TST/None/")
    print(f"   {root}/results/TransAppResults_TST/Embed/")